# 06: Evaluation Protocol

Section 3.5's shared evaluation setup for comparing the five representations (`raw`, `classical`, `ft_transformer`, `subtab`, `scarf`) built in `04_feature_encoding.ipynb`/`05_embedding_paradigms.ipynb`.

## Setup

In [ ]:
%matplotlib inline

import os

# torch bundles its own OpenMP runtime (libiomp5md.dll) while
# numpy/pandas/scikit-learn pull in MKL's (libomp.dll/libiomp5md.dll) --
# loading both into one process aborts the kernel outright ("OMP: Error
# #15: Initializing ..., but found ... already initialized"), not a
# Python exception, so it shows up as a bare kernel crash with no
# traceback. Must be set before numpy/torch are imported anywhere below.
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# silhouette_score (src/evaluation/clustering_metrics.py) segfaults this
# process outright on wide representation matrices under BLAS's default
# multi-threaded configuration -- confirmed reproducible, documented in
# clustering_metrics.py's own module docstring. Must be set before numpy
# is imported anywhere below, same placement discipline as the
# KMP_DUPLICATE_LIB_OK line above.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import sys
from pathlib import Path

from IPython.display import display

import lime.lime_tabular
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import xgboost

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import config
from src.evaluation import eval_setup

sns.set_theme(style="whitegrid")

ARTIFACTS_DIR = REPO_ROOT / "data" / "artifacts" / eval_setup.ACTIVE_CANDIDATE
FINAL_DIR = config.FINAL_DIR

## 1. Load representations and candidate splits

The one cell every later cell in this notebook depends on. Loads all five representations (`eval_setup.load_representations`) and the candidate's own split files (`eval_setup.load_candidate_splits`), then asserts row alignment between them (`eval_setup.assert_row_alignment`) before anything downstream trusts positional alignment between a representation frame and `candidate_splits`. Left uncaught -- if alignment is wrong, this cell must fail loudly, not be swallowed and discovered later as a silently wrong score.

In [ ]:
representations = eval_setup.load_representations(ARTIFACTS_DIR)
candidate_splits = eval_setup.load_candidate_splits(FINAL_DIR)

eval_setup.assert_row_alignment(representations, candidate_splits)

for repr_name in eval_setup.REPR_NAMES:
    for split in eval_setup.SPLITS:
        shape = representations[repr_name][split].shape
        print(f"representations[{repr_name!r}][{split!r}]: {shape}")

for split in eval_setup.SPLITS:
    print(f"candidate_splits[{split!r}]: {candidate_splits[split].shape}")

## 2. Derive the genre label

`eval_setup.derive_genre_label` (first-listed genre; empty-list rows fall back to the training split's own most frequent first-listed genre). The fallback-row count it already prints per split is re-checked here independently, straight from `candidate_splits`' own `genres` column, rather than just trusting that prior print -- the two counts should agree exactly.

In [ ]:
genre_labels = eval_setup.derive_genre_label(candidate_splits)

print("genre_labels['test'].value_counts():")
print(genre_labels["test"].value_counts())
print()

for split in eval_setup.SPLITS:
    n_distinct = genre_labels[split].nunique()
    print(f"{split}: {n_distinct} distinct genres")
print()

# Independent re-check of the fallback count derive_genre_label
# already printed -- recomputed here directly from
# candidate_splits' own genres column (empty list, not NaN), not
# trusted from that prior cell's stdout.
for split in eval_setup.SPLITS:
    is_empty = candidate_splits[split][eval_setup.GENRE_COL].apply(len) == 0
    print(f"{split}: {is_empty.sum()} row(s) had an empty genres list (fallback expected)")

## 3. Shared XGBoost config and sensitive-attribute-safe features

`eval_setup.SharedXGBConfig` -- the one fixed, un-tuned config every representation is scored with (its own docstring covers why). Printed here so the exact hyperparameters feeding every later step are visible in this notebook's output, not just in source.

Also builds `raw_features_safe`/`classical_features_safe`: `eval_setup.drop_sensitive_attribute_columns` applied to `representations["raw"]`/`representations["classical"]` for every split -- [D46]'s guard against the recoverability probe trivially predicting `original_language` from a column that IS `original_language`.

In [ ]:
import dataclasses

xgb_config = eval_setup.SharedXGBConfig()

print("dataclasses.asdict(xgb_config):")
print(dataclasses.asdict(xgb_config))
print()
print("xgb_config.regressor_kwargs():")
print(xgb_config.regressor_kwargs())
print()
print("xgb_config.classifier_kwargs():")
print(xgb_config.classifier_kwargs())

raw_features_safe = {
    split: eval_setup.drop_sensitive_attribute_columns(representations["raw"][split], "raw")
    for split in eval_setup.SPLITS
}
classical_features_safe = {
    split: eval_setup.drop_sensitive_attribute_columns(
        representations["classical"][split], "classical"
    )
    for split in eval_setup.SPLITS
}

raw_before = representations["raw"]["test"].shape[1]
raw_after = raw_features_safe["test"].shape[1]
classical_before = representations["classical"]["test"].shape[1]
classical_after = classical_features_safe["test"].shape[1]

print()
print(f"raw (test split): {raw_before} columns before -> {raw_after} after (dropped {raw_before - raw_after})")
print(
    f"classical (test split): {classical_before} columns before -> "
    f"{classical_after} after (dropped {classical_before - classical_after})"
)

## 4. Clustering metrics (Eq. 3.16-3.18)

`clustering_metrics.run_all` against the test split, for every representation in `eval_setup.REPR_NAMES`. Uses the ORIGINAL `representations["raw"]`/`representations["classical"]` frames here, not `raw_features_safe`/`classical_features_safe` from Section 3 -- dropping `original_language` is only required for the Step 6 fairness/recoverability probe, not for clustering quality, so there is no reason to weaken raw/classical's own feature set here.

In [ ]:
from src.evaluation import clustering_metrics

repr_arrays = {
    repr_name: representations[repr_name]["test"].to_numpy()
    for repr_name in eval_setup.REPR_NAMES
}

clustering_table = clustering_metrics.run_all(
    repr_arrays, genre_labels["test"], eval_setup.RANDOM_SEED
)
display(clustering_table)

### Save clustering metrics

Sanity-checks `clustering_table` before saving -- every `silhouette` value must be in `[-1, 1]` and every `davies_bouldin` value must be `>= 0` (per each metric's own definition, `clustering_metrics.py`'s own docstring), so a metric accidentally computed on the wrong axis or against the wrong label array fails loudly here rather than silently propagating into Table 3.3 later. Saved to `ARTIFACTS_DIR / "evaluation" / "clustering_metrics.csv"`, the same `ARTIFACTS_DIR / <subfolder>` convention as the `"models" / paradigm / "diagnostics"` save paths used elsewhere in this codebase.

In [ ]:
assert clustering_table["silhouette"].between(-1, 1).all(), (
    "silhouette values out of [-1, 1] range:\n"
    f"{clustering_table.loc[~clustering_table['silhouette'].between(-1, 1), 'silhouette']}"
)
assert (clustering_table["davies_bouldin"] >= 0).all(), (
    "davies_bouldin values below 0:\n"
    f"{clustering_table.loc[clustering_table['davies_bouldin'] < 0, 'davies_bouldin']}"
)
print("Sanity checks passed: silhouette in [-1, 1], davies_bouldin >= 0")

EVALUATION_DIR = ARTIFACTS_DIR / "evaluation"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

clustering_metrics_path = EVALUATION_DIR / "clustering_metrics.csv"
clustering_table.to_csv(clustering_metrics_path)
print(f"Saved: {clustering_metrics_path}")

## 5. t-SNE parameter check (Kobak & Berens 2019 scaling rule)

`tsne_viz.kobak_berens_tsne_params` against the real test-split size, printed alongside sklearn's own `TSNE` defaults (`perplexity=30`, `learning_rate="auto"`) so the notebook's own output makes the "validated, not default" claim checkable at a glance, per the code-comment requirement in `Chapter3_Methodology_Structure.md`'s SS3.5 visualisation bullet. At this split size both floors bind (`1051/100 ~= 10.5 < 30`, `1051/12 ~= 87.6 < 200`), so `perplexity` lands on 30 -- the same NUMBER as sklearn's own default, but not the same thing: it's the floor of a size-aware rule, not an unscaled constant, and would scale up for a larger split.

In [ ]:
from src.evaluation import tsne_viz

n_test = len(genre_labels["test"])
tsne_params = tsne_viz.kobak_berens_tsne_params(n_test)

print(f"n_test = {n_test}")
print(f"kobak_berens_tsne_params({n_test}) = {tsne_params}")
print()
print(f"  n_test / 100 = {n_test / 100:.2f}  (perplexity floor 30 binds: {n_test / 100 < 30})")
print(f"  n_test / 12  = {n_test / 12:.2f}  (learning_rate floor 200 binds: {n_test / 12 < 200})")
print()
print("Comparison against sklearn's own TSNE(...) defaults:")
print(f"  perplexity:    kobak_berens={tsne_params['perplexity']!r}   vs  sklearn default=30")
print(f"  learning_rate: kobak_berens={tsne_params['learning_rate']!r}  vs  sklearn default='auto'")

assert tsne_params["perplexity"] == 30, (
    f"expected perplexity=30 (floor should bind at n_test={n_test}), "
    f"got {tsne_params['perplexity']}"
)
assert tsne_params["learning_rate"] == 200, (
    f"expected learning_rate=200 (floor should bind at n_test={n_test}), "
    f"got {tsne_params['learning_rate']}"
)
print("\nBoth floors confirmed binding at this split size, as expected.")

## 6. Fit t-SNE per representation

**The slowest cell in this notebook** -- fits `tsne_viz.fit_tsne` once per `eval_setup.REPR_NAMES` representation on the test split, storing results only. No plotting here, deliberately, so a slow re-run for debugging the plot doesn't force re-fitting t-SNE every time.

`max_samples=100` is passed explicitly -- NOT optional here: `tsne_viz.py`'s own module docstring documents that even the combination of fixes already baked into `fit_tsne` (`method="exact"`, `n_jobs=1`) only holds up to N=100 on real data in this environment; the real test split has 1051 rows, and calling `fit_tsne` without `max_samples` would crash the kernel. 100 is the largest value that investigation confirmed safe.

Stores BOTH `tsne_embeddings[repr_name]` (the `(100, 2)` layout) and `tsne_sample_indices[repr_name]` (which of the 1051 test rows those 100 points came from) -- `fit_tsne` always returns both together specifically so a later cell can align `genre_labels["test"]` to the subsampled points correctly via `genre_labels["test"].to_numpy()[tsne_sample_indices[repr_name]]`, not just the embedding alone.

In [ ]:
import time

from src.evaluation import tsne_viz

tsne_embeddings = {}
tsne_sample_indices = {}

for repr_name in eval_setup.REPR_NAMES:
    X = representations[repr_name]["test"].to_numpy()

    start = time.time()
    embedding, sample_indices = tsne_viz.fit_tsne(
        X, eval_setup.RANDOM_SEED, max_samples=100
    )
    elapsed = time.time() - start

    tsne_embeddings[repr_name] = embedding
    tsne_sample_indices[repr_name] = sample_indices
    print(f"{repr_name}: {elapsed:.1f}s")

## 7. t-SNE grid

One panel per `eval_setup.REPR_NAMES` representation, each a scatter of that representation's `tsne_embeddings[repr]` coloured by `genre_labels["test"]` -- points are aligned back to their genre via `tsne_sample_indices[repr]` (Section 6), not `genre_labels["test"]` directly, since each panel only plots its own 100-row subsample. One genre-to-colour mapping is built ONCE and reused for all five panels (fixed order, never recycled per panel), so the same genre is the same colour everywhere and panels are actually comparable. Saved via `src/eda/report.py`'s own `savefig` (300dpi, `bbox_inches="tight"`, same convention already used in `00_tmdb_eda.ipynb`/`01_overview_cleaning.ipynb`/`03_feature_split.ipynb`) rather than reimplementing the same save logic here.

In [ ]:
from src.eda.report import savefig

genres_sorted = sorted(genre_labels["test"].unique())
palette = sns.color_palette("tab20", n_colors=len(genres_sorted))
genre_to_color = dict(zip(genres_sorted, palette))

test_labels_array = genre_labels["test"].to_numpy()

fig, axes = plt.subplots(
    1, len(eval_setup.REPR_NAMES), figsize=(5 * len(eval_setup.REPR_NAMES), 5)
)

for ax, repr_name in zip(axes, eval_setup.REPR_NAMES):
    embedding = tsne_embeddings[repr_name]
    panel_labels = test_labels_array[tsne_sample_indices[repr_name]]
    colors = [genre_to_color[g] for g in panel_labels]

    ax.scatter(embedding[:, 0], embedding[:, 1], c=colors, s=20, edgecolor="none")
    ax.set_title(repr_name)
    ax.set_xticks([])
    ax.set_yticks([])

legend_handles = [
    plt.Line2D([0], [0], marker="o", linestyle="", color=genre_to_color[g], label=g)
    for g in genres_sorted
]
fig.legend(
    handles=legend_handles, loc="lower center", ncol=min(len(genres_sorted), 9),
    bbox_to_anchor=(0.5, -0.1), title="genre",
)

fig.suptitle("t-SNE projections (illustrative, not evidential) \u2014 genre label from Step 1")
fig.tight_layout()

TSNE_FIGURES_DIR = ARTIFACTS_DIR / "evaluation" / "figures"
savefig(fig, "tsne_grid.png", figures_dir=TSNE_FIGURES_DIR)
plt.show()

## 8. Fit host regressors (RQ2/DO3 -- NOT a DO2 quality axis)

`host_model.fit_host_regressor` per representation, on the SAME `xgb_config` (Section 3) every other XGBoost fit in this notebook uses -- no early stopping, no per-representation tuning, per [D12]. These models exist only so Step 5's LIME/SHAP have something to explain; see `host_model.py`'s own module docstring for why this is explicitly not a representation-quality comparison.

In [ ]:
from src.evaluation import host_model

host_models = {}
for repr_name in eval_setup.REPR_NAMES:
    X_train = representations[repr_name]["train"].to_numpy()
    y_train = candidate_splits["train"][eval_setup.TARGET_COL].to_numpy()
    host_models[repr_name] = host_model.fit_host_regressor(X_train, y_train, xgb_config)
    print(f"{repr_name}: fitted")

### Evaluate host regressors

`host_model.evaluate_host` per representation on the test split, collected into `host_scores`. The circularity caveat printed immediately after is pulled straight from `host_model.py`'s own module docstring (not retyped here), so this notebook's output always carries whatever the .py file's current wording actually says, never a copy that can silently drift out of sync with it.

In [ ]:
host_scores_rows = {}
for repr_name in eval_setup.REPR_NAMES:
    X_test = representations[repr_name]["test"].to_numpy()
    y_test = candidate_splits["test"][eval_setup.TARGET_COL].to_numpy()
    host_scores_rows[repr_name] = host_model.evaluate_host(
        host_models[repr_name], X_test, y_test
    )

host_scores = pd.DataFrame.from_dict(host_scores_rows, orient="index")
display(host_scores)

## 9. Fixed sample for LIME

LIME's per-instance surrogate fitting is the slow path in `src/evaluation/interpretability.py` (`lime_explain_sample`'s own docstring), so Section 3.5's LIME comparison runs on a fixed 100-row sample of the test split rather than all 1051 rows. `lime_sample_idx` is drawn once here, with `eval_setup.RANDOM_SEED`, and sorted so row order stays consistent with the split's own order. Because every representation's test frame shares `candidate_splits["test"]`'s row order (`assert_row_alignment`, Section 1), the SAME `lime_sample_idx` indexes correctly into any `representations[repr_name]["test"]` -- there is nothing representation-specific to re-derive.

In [ ]:
lime_sample_idx = np.sort(
    np.random.default_rng(eval_setup.RANDOM_SEED).choice(
        len(candidate_splits["test"]), size=100, replace=False
    )
)

print(f"lime_sample_idx: {len(lime_sample_idx)} rows sampled (seed={eval_setup.RANDOM_SEED})")
print(
    "Every representation's test frame shares candidate_splits['test']'s row "
    "order (Section 1's assert_row_alignment already confirmed this), so "
    "lime_sample_idx indexes correctly into representations[repr_name]['test'] "
    "for every repr_name -- no per-representation re-derivation needed."
)

## 10. SHAP explanations

`interpretability.shap_explain` per representation, on the full test split (`shap.TreeExplainer` is fast enough not to need `lime_sample_idx`'s subsampling -- see `interpretability.py`'s own docstring on why TreeExplainer is a legitimate implementation-level shortcut, not a different quantity). Also builds `feature_names` per representation here, since Step 11's LIME explanations and Step 12's summaries both need real column names for `raw`/`classical` and placeholder `dim_i` names for the three embeddings -- widths differ per representation, so this cannot be a single shared list.

In [ ]:
import time

from src.evaluation import interpretability

shap_values = {}
feature_names = {}

for repr_name in eval_setup.REPR_NAMES:
    X_test = representations[repr_name]["test"].to_numpy()

    if repr_name in ("raw", "classical"):
        feature_names[repr_name] = representations[repr_name]["test"].columns.tolist()
    else:
        feature_names[repr_name] = [f"dim_{i}" for i in range(X_test.shape[1])]

    start = time.time()
    shap_values[repr_name] = interpretability.shap_explain(host_models[repr_name], X_test)
    elapsed = time.time() - start
    print(f"{repr_name}: {elapsed:.1f}s ({X_test.shape[1]} features)")

## 11. LIME explanations

`interpretability.lime_explain_sample` per representation, on `lime_sample_idx`'s fixed 100-row test subsample (Section 9) -- LIME fits one local surrogate model per row, so this is expected to be the slowest cell in the notebook after Section 6's t-SNE fit, worst-case on `classical`'s 4282 columns. A representation that takes more than a few minutes prints a warning but is left to finish -- silently truncating the sample would mean a different set of rows gets explained per representation, breaking the row-for-row SHAP/LIME comparison Section 12 needs.

In [ ]:
import time

LIME_SLOW_WARNING_SECONDS = 180

lime_explanations = {}

for repr_name in eval_setup.REPR_NAMES:
    X_train = representations[repr_name]["train"].to_numpy()
    X_sample = representations[repr_name]["test"].to_numpy()[lime_sample_idx]

    start = time.time()
    lime_explanations[repr_name] = interpretability.lime_explain_sample(
        host_models[repr_name],
        X_train,
        X_sample,
        feature_names[repr_name],
        eval_setup.RANDOM_SEED,
    )
    elapsed = time.time() - start

    if elapsed > LIME_SLOW_WARNING_SECONDS:
        print(
            f"WARNING: {repr_name} took {elapsed:.1f}s (> {LIME_SLOW_WARNING_SECONDS}s) -- "
            "letting it finish rather than truncating the sample."
        )
    print(f"{repr_name}: {elapsed:.1f}s")

## 12. SHAP/LIME agreement

`interpretability.summarize_shap`, `interpretability.summarize_lime`, and `interpretability.attribution_agreement` ([D49]) per representation, collected into `interpretability_table`. The single agreement number is a Table 3.3 trade-off summary only (see `attribution_agreement`'s own docstring) -- it does not by itself show WHERE the two methods disagree, so each representation's top-5 SHAP and top-5 LIME features are also printed side by side, and the representation with the lowest agreement score gets an explicit line naming which features are top-5 in one method but missing from the other's top-10. That is the qualitative disagreement Section 3.5's prose needs to discuss.

In [ ]:
interpretability_rows = {}
top_shap = {}
top_lime = {}

for repr_name in eval_setup.REPR_NAMES:
    shap_summary = interpretability.summarize_shap(shap_values[repr_name], feature_names[repr_name])
    lime_summary = interpretability.summarize_lime(lime_explanations[repr_name])
    agreement = interpretability.attribution_agreement(shap_summary, lime_summary)

    interpretability_rows[repr_name] = {"attribution_agreement": agreement}
    top_shap[repr_name] = shap_summary
    top_lime[repr_name] = lime_summary

interpretability_table = pd.DataFrame.from_dict(interpretability_rows, orient="index")
display(interpretability_table)

for repr_name in eval_setup.REPR_NAMES:
    agreement = interpretability_table.loc[repr_name, "attribution_agreement"]
    print(f"\n{repr_name} (agreement={agreement:.3f}):")
    print(f"  top-5 SHAP: {top_shap[repr_name].head(5).index.tolist()}")
    print(f"  top-5 LIME: {top_lime[repr_name].head(5).index.tolist()}")

lowest_repr = interpretability_table["attribution_agreement"].idxmin()
shap_top5 = set(top_shap[lowest_repr].head(5).index)
shap_top10 = set(top_shap[lowest_repr].index)
lime_top5 = set(top_lime[lowest_repr].head(5).index)
lime_top10 = set(top_lime[lowest_repr].index)

disagreeing = (shap_top5 - lime_top10) | (lime_top5 - shap_top10)
print(
    f"\nLowest agreement: {lowest_repr} "
    f"(score={interpretability_table.loc[lowest_repr, 'attribution_agreement']:.3f}) -- "
    "feature(s) top-5 in one method but absent from the other's top-10: "
    f"{sorted(disagreeing) if disagreeing else 'none'}"
)

## 13. Shared language label encoding

One `sklearn.preprocessing.LabelEncoder`, fit once on `candidate_splits["train"]["original_language"]` and reused for every split and every representation's Step 14 recoverability probe -- a single shared encoder, not one per representation, so every probe's confusion matrix uses IDENTICAL class indices and which classes each representation confuses is actually comparable across representations, not just the aggregate accuracy/macro_f1 numbers.

Asserts every value in `candidate_splits["test"]["original_language"]` was already seen when the encoder was fit on train, before transforming test -- if this ever fails, it means the D26 rare-language-bucketing assumption (`03_feature_split.ipynb`) was not applied consistently across splits, which needs re-checking there, not silently working around here (e.g. by adding an "unknown" class after the fact).

Prints `candidate_splits["train"]["original_language"].value_counts()` so the class imbalance `fairness.evaluate_probe`'s own docstring warns about is visible before interpreting any accuracy/macro_f1 pair later.</cell id="f5042869">


In [ ]:
from sklearn.preprocessing import LabelEncoder

print("candidate_splits['train']['original_language'].value_counts():")
print(candidate_splits["train"][eval_setup.SENSITIVE_COL].value_counts())

lang_encoder = LabelEncoder()
lang_encoder.fit(candidate_splits["train"][eval_setup.SENSITIVE_COL])

unseen_test_langs = set(candidate_splits["test"][eval_setup.SENSITIVE_COL].unique()) - set(
    lang_encoder.classes_
)
assert not unseen_test_langs, (
    f"candidate_splits['test']['original_language'] contains value(s) never seen in train: "
    f"{unseen_test_langs} -- the D26 rare-language-bucketing assumption "
    "(03_feature_split.ipynb) was not applied consistently across splits and needs "
    "re-checking there, not silently working around here"
)

y_lang = {
    split: lang_encoder.transform(candidate_splits[split][eval_setup.SENSITIVE_COL])
    for split in eval_setup.SPLITS
}

print()
print(f"lang_encoder.classes_: {lang_encoder.classes_.tolist()}")
for split in eval_setup.SPLITS:
    print(f"y_lang[{split!r}]: {y_lang[split].shape}")

## 14. Fit recoverability probes (Song & Raghunathan 2020)

`fairness.fit_recoverability_probe` per representation, on `y_lang["train"]` (Section 13) and the SAME `xgb_config` (Section 3) every other XGBoost fit in this notebook uses -- per [D45], no per-representation tuning. `raw`/`classical` use `raw_features_safe`/`classical_features_safe` (Section 3, [D46]'s sensitive-attribute-safe columns), not `representations["raw"]`/`representations["classical"]` directly -- fitting on the unsafe versions would let the probe trivially recover `original_language` from a column that IS `original_language`. `ft_transformer`/`subtab`/`scarf` use `representations[repr]` unchanged, since those are opaque embeddings with no such column to drop.</cell id="ded37903">


In [ ]:
from src.evaluation import fairness

fairness_probes = {}

for repr_name in eval_setup.REPR_NAMES:
    if repr_name == "raw":
        X_train = raw_features_safe["train"].to_numpy()
    elif repr_name == "classical":
        X_train = classical_features_safe["train"].to_numpy()
    else:
        X_train = representations[repr_name]["train"].to_numpy()

    fairness_probes[repr_name] = fairness.fit_recoverability_probe(
        X_train, y_lang["train"], xgb_config
    )
    print(f"{repr_name}: fitted")
    if repr_name in ("raw", "classical"):
        print(f"  {repr_name} feature count used: {X_train.shape[1]}")

### Evaluate recoverability probes and amplification (Eq. 3.20)

`fairness.evaluate_probe` per representation on the same leakage-safe/unchanged test matrices as the fit cell above, collected into `fairness_table`. `delta_accuracy`/`delta_macro_f1` are `fairness.compute_amplification` against `raw`'s own scores -- left `NaN` for the `raw` row itself, since amplification is only defined relative to raw, not raw relative to itself. Positive delta = amplified, negative = obscured (the Zemel et al. 2013 converse case; see `fairness.compute_amplification`'s own docstring) -- printed explicitly per representation using `delta_accuracy` as the primary reading, flagging any representation where `delta_macro_f1` disagrees in sign, since that disagreement is itself worth a line in Section 3.5's prose, not just a number in the table.</cell id="ee0a470e">


In [ ]:
fairness_rows = {}

for repr_name in eval_setup.REPR_NAMES:
    if repr_name == "raw":
        X_test = raw_features_safe["test"].to_numpy()
    elif repr_name == "classical":
        X_test = classical_features_safe["test"].to_numpy()
    else:
        X_test = representations[repr_name]["test"].to_numpy()

    fairness_rows[repr_name] = fairness.evaluate_probe(
        fairness_probes[repr_name], X_test, y_lang["test"]
    )

fairness_table = pd.DataFrame.from_dict(fairness_rows, orient="index")
fairness_table["delta_accuracy"] = np.nan
fairness_table["delta_macro_f1"] = np.nan

raw_accuracy = fairness_table.loc["raw", "accuracy"]
raw_macro_f1 = fairness_table.loc["raw", "macro_f1"]

for repr_name in eval_setup.REPR_NAMES:
    if repr_name == "raw":
        continue
    fairness_table.loc[repr_name, "delta_accuracy"] = fairness.compute_amplification(
        fairness_table.loc[repr_name, "accuracy"], raw_accuracy
    )
    fairness_table.loc[repr_name, "delta_macro_f1"] = fairness.compute_amplification(
        fairness_table.loc[repr_name, "macro_f1"], raw_macro_f1
    )

display(fairness_table)

for repr_name in eval_setup.REPR_NAMES:
    if repr_name == "raw":
        continue
    delta_acc = fairness_table.loc[repr_name, "delta_accuracy"]
    delta_f1 = fairness_table.loc[repr_name, "delta_macro_f1"]

    if delta_acc > 0:
        direction = "amplifies"
    elif delta_acc < 0:
        direction = "obscures"
    else:
        direction = "neither amplifies nor obscures (delta_accuracy == 0)"

    sentence = (
        f"{repr_name} {direction} the sensitive attribute relative to raw "
        f"(delta_accuracy={delta_acc:+.4f})"
    )

    if np.sign(delta_acc) != np.sign(delta_f1):
        sentence += (
            f" -- but delta_macro_f1={delta_f1:+.4f} disagrees in sign, "
            "worth a line in 3.5's prose"
        )
    print(sentence)